# Lesson 6: Behavior

**You will learn:** how a benchmark asks a subject for behavioral answers, and the two ways a `BrainScoreModel` can produce them.
**Requires:** CPU only, no downloads, under a minute.

Neural benchmarks compare activity. Behavioral benchmarks compare choices, such as which of two labels a participant picks for an image. The task is described by a `TaskContext`:

| Field | Meaning |
| --- | --- |
| `task_type` | The kind of answer, for example `'probabilities'` (a probability per label) or `'label'`. |
| `label_set` | The answers the benchmark accepts. |
| `fitting_stimuli` | Labeled examples, used when the model must learn the task first. |
| `instruction` | A text instruction, used when the model can follow instructions. |

A `BrainScoreModel` can answer in two ways:

- **Readout:** fit a simple classifier on the activity of one layer (`behavioral_readout_layer`) using `fitting_stimuli`. This suits networks that were never trained on the task.
- **Answer:** call your own `answer(stimulus_row, instruction, label_set)` function, for example a language model that generates a reply. This suits instruction-following models.

These are different measurements. A benchmark that depends on one of them selects it with `prefer_path='readout'` or `prefer_path='generation'` (the path that calls `answer`).

In [1]:
import warnings
warnings.filterwarnings('ignore')  # keep the lesson output short

In [2]:
import tempfile, pathlib
import numpy as np
from PIL import Image
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# Eight small synthetic images: four bright, four dark. They stand in for real stimuli.
folder = pathlib.Path(tempfile.mkdtemp())
rng = np.random.default_rng(0)
rows = []
for i in range(8):
    bright = i % 2 == 0
    low, high = (150, 255) if bright else (0, 100)
    path = folder / f'image_{i}.png'
    Image.fromarray(rng.integers(low, high, (16, 16, 3), dtype=np.uint8)).save(path)
    rows.append({'stimulus_id': f'image_{i}', 'image_path': str(path),
                 'label': 'bright' if bright else 'dark'})

stimuli = StimulusSet(rows)
stimuli.identifier = 'toy-bright-dark'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli

,stimulus_id,image_path,label
0,image_0,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
1,image_1,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
2,image_2,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
3,image_3,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
4,image_4,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
5,image_5,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
6,image_6,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
7,image_7,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark


In [3]:
import torch

torch.manual_seed(0)
# A tiny, untrained convolutional network. Its weights are random.
net = torch.nn.Sequential(
    torch.nn.Conv2d(3, 4, kernel_size=3),  # layer '0'
    torch.nn.ReLU(),                       # layer '1'
    torch.nn.AdaptiveAvgPool2d(2),         # layer '2'
    torch.nn.Flatten(),                    # layer '3'
    torch.nn.Linear(16, 8),                # layer '4'
).eval()

def preprocess(paths):
    """Load image files into a (batch, 3, height, width) float array."""
    images = [np.asarray(Image.open(p).convert('RGB'), dtype=np.float32) / 255 for p in paths]
    return np.stack(images).transpose(0, 3, 1, 2)

## Readout

In [4]:
from brainscore_core import BrainScoreModel, TaskContext
from brainscore_core.streaming import BufferedSession
from brainscore.model_helpers.vision_wrapper import VisionWrapper

model = BrainScoreModel(
    identifier='toy-cnn',
    model=net,
    extractor={'vision': VisionWrapper(identifier='toy-cnn', model=net, preprocessing=preprocess)},
    behavioral_readout_layer='4',
)

task = TaskContext(task_type='probabilities', label_set=['bright', 'dark'],
                   fitting_stimuli=stimuli, prefer_path='readout')
session = BufferedSession(task_context=task, requested_output_channels=['behavior'])
model.participate(session)
choices = session.collect()
model.reset()
choices

activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

<xarray.BehavioralAssembly (presentation: 8, choice: 2)>
array([[0.5147406 , 0.48525944],
       [0.4814522 , 0.51854783],
       [0.5144072 , 0.4855928 ],
       [0.48630482, 0.5136952 ],
       [0.5137604 , 0.4862396 ],
       [0.48849556, 0.5115045 ],
       [0.514655  , 0.48534495],
       [0.48618582, 0.5138142 ]], dtype=float32)
Coordinates:
  * choice                        (choice) <U6 'bright' 'dark'
  * presentation                  (presentation) MultiIndex
  - microsaccade_shift_x_degrees  (presentation) float64 0.0 0.0 0.0 ... 0.0 0.0
  - microsaccade_shift_y_degrees  (presentation) float64 0.0 0.0 0.0 ... 0.0 0.0
  - microsaccade_shift_y_pixels   (presentation) float64 0.0 0.0 0.0 ... 0.0 0.0
  - label                         (presentation) object 'bright' ... 'dark'
  - image_path                    (presentation) object '/var/folders/h6/g53m...
  - stimulus_id                   (presentation) object 'image_0' ... 'image_7'
  - microsaccade_shift_x_pixels   (presentation) float64 0.0 0.0 0.0 ... 0.0 0.0

The result is a `BehavioralAssembly` with one row per stimulus and one column per label. For this lesson the readout is fitted and tested on the same eight images, so the numbers say nothing about generalization; a real benchmark keeps fitting and test stimuli separate.

## Answer

An `answer` function receives one stimulus row, the instruction and the label set, and returns one label. The rule below is a stand-in for a model that looks at the image and replies in words.

In [5]:
def answer(stimulus_row, instruction, label_set):
    brightness = np.asarray(Image.open(stimulus_row['image_path'])).mean()
    return 'bright' if brightness > 127 else 'dark'

answering_model = BrainScoreModel(identifier='brightness-judge', supported_modalities={'vision'}, answer=answer)

task = TaskContext(task_type='label', label_set=['bright', 'dark'],
                   instruction='Is this image bright or dark?', fitting_stimuli=stimuli)
session = BufferedSession(task_context=task, requested_output_channels=['behavior'])
answering_model.participate(session)
session.collect()

<xarray.BehavioralAssembly (presentation: 8, choice: 2)>
array([[1., 0.],
       [0., 1.],
       [1., 0.],
       [0., 1.],
       [1., 0.],
       [0., 1.],
       [1., 0.],
       [0., 1.]], dtype=float32)
Coordinates:
  * choice        (choice) <U6 'bright' 'dark'
  * presentation  (presentation) MultiIndex
  - stimulus_id   (presentation) object 'image_0' 'image_1' ... 'image_7'
  - image_path    (presentation) object '/var/folders/h6/g53mxg516n538136hh74...
  - label         (presentation) object 'bright' 'dark' ... 'bright' 'dark'

`supported_modalities={'vision'}` declares the input, because this subject has no extractor. It has no layers, so it can produce behavior but not neural activity.

## Summary

- A `TaskContext` describes the behavioral task.
- **Readout** fits a classifier on recorded activity; **answer** calls your own function.
- Choose the path explicitly when the distinction matters for your scientific claim.

**Next:** [Lesson 7](07_perturb_a_model.ipynb) changes the model's activity during a run.